# Detection Baseline — VisDrone2019-DET

**Course:** Computer Vision Engineering for DefenceTech  
**Assignment:** Detection Baseline Report (Part 1 of Detection & Error Analysis Report)  
**Author:** Serhii Polishchuk  

This notebook:
1. Sets up a reproducible **baseline** detector (YOLOv8n)
2. Computes mAP50-95, precision, recall
3. Analyses quality on small / medium / large objects
4. Runs **one controlled change** (higher input resolution)
5. Compares before/after (quality, small-object quality, time, memory)

## 0. Setup

In [ ]:
# Install dependencies (Colab)
!pip install -q ultralytics opencv-python-headless

import os
import time
import json
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image

from ultralytics import YOLO
import torch

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    HARDWARE = f"GPU: {torch.cuda.get_device_name(0)}"
else:
    HARDWARE = "CPU"
print("Hardware:", HARDWARE)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
Hardware: GPU: Tesla T4


## 1. Dataset paths

We reuse VisDrone2019-DET that was downloaded in DZ01.
If the dataset is not present, the download cell below will fetch it.

In [ ]:
DATA_ROOT = Path("./datasets/VisDrone")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# Official Ultralytics mirrors (same as DZ01)
URLS = {
    "VisDrone2019-DET-train": "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip",
    "VisDrone2019-DET-val":   "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip",
    "VisDrone2019-DET-test-dev": "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-test-dev.zip",
}

import zipfile
import urllib.request

def download_and_extract(name: str, url: str, root: Path):
    target_dir = root / name
    zip_path = root / f"{name}.zip"
    if target_dir.exists() and any(target_dir.rglob("*.jpg")):
        print(f"[SKIP] {name} already exists")
        return
    print(f"[DOWNLOAD] {name} ...")
    urllib.request.urlretrieve(url, zip_path)
    print(f"[EXTRACT]  {name} ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(root)
    zip_path.unlink(missing_ok=True)
    print(f"[DONE]    {name}")

for name, url in URLS.items():
    download_and_extract(name, url, DATA_ROOT)

print("\nDataset root contents:")
for p in sorted(DATA_ROOT.iterdir()):
    print(" -", p.name)

[DOWNLOAD] VisDrone2019-DET-train ...
[EXTRACT]  VisDrone2019-DET-train ...
[DONE]    VisDrone2019-DET-train
[DOWNLOAD] VisDrone2019-DET-val ...
[EXTRACT]  VisDrone2019-DET-val ...
[DONE]    VisDrone2019-DET-val
[DOWNLOAD] VisDrone2019-DET-test-dev ...
[EXTRACT]  VisDrone2019-DET-test-dev ...
[DONE]    VisDrone2019-DET-test-dev

Dataset root contents:
 - VisDrone2019-DET-test-dev
 - VisDrone2019-DET-train
 - VisDrone2019-DET-val


## 2. Convert VisDrone annotations → YOLO format

VisDrone original format:  
`<x>,<y>,<w>,<h>,<score>,<category>,<truncation>,<occlusion>`  

YOLO format:  
`<class> <x_center> <y_center> <width> <height>` (normalized 0–1)  

We drop class 0 (ignored regions) and shift class ids by -1.

In [ ]:
from tqdm.auto import tqdm

CLASS_NAMES = {
    0: "pedestrian",
    1: "people",
    2: "bicycle",
    3: "car",
    4: "van",
    5: "truck",
    6: "tricycle",
    7: "awning-tricycle",
    8: "bus",
    9: "motor",
}

def convert_visdrone_to_yolo(split_dir: Path):
    """Convert one VisDrone split to YOLO labels."""
    images_dir = split_dir / "images"
    ann_dir = split_dir / "annotations"
    labels_dir = split_dir / "labels"
    labels_dir.mkdir(exist_ok=True)

    ann_files = list(ann_dir.glob("*.txt"))
    converted = 0

    for ann_path in tqdm(ann_files, desc=f"Converting {split_dir.name}"):
        img_path = images_dir / (ann_path.stem + ".jpg")
        if not img_path.exists():
            continue

        with Image.open(img_path) as im:
            W, H = im.size

        lines_out = []
        with open(ann_path) as f:
            for row in f:
                parts = row.strip().split(",")
                if len(parts) < 6:
                    continue
                x, y, w, h = map(int, parts[:4])
                cat = int(parts[5])

                # skip ignored regions (class 0) and "others" (class 11)
                if cat == 0 or cat == 11:
                    continue
                if w <= 0 or h <= 0:
                    continue

                cls = cat - 1  # shift to 0..9
                xc = (x + w / 2) / W
                yc = (y + h / 2) / H
                bw = w / W
                bh = h / H

                # clamp
                xc = min(max(xc, 0.0), 1.0)
                yc = min(max(yc, 0.0), 1.0)
                bw = min(max(bw, 0.0), 1.0)
                bh = min(max(bh, 0.0), 1.0)

                lines_out.append(f"{cls} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

        out_path = labels_dir / (ann_path.stem + ".txt")
        with open(out_path, "w") as f:
            f.writelines(lines_out)
        converted += 1

    print(f"  → {converted} label files written to {labels_dir}")
    return converted

for split_name in ["VisDrone2019-DET-train", "VisDrone2019-DET-val", "VisDrone2019-DET-test-dev"]:
    split_path = DATA_ROOT / split_name
    if split_path.exists():
        convert_visdrone_to_yolo(split_path)

Converting VisDrone2019-DET-train:   0%|          | 0/6471 [00:00<?, ?it/s]

  → 6471 label files written to datasets/VisDrone/VisDrone2019-DET-train/labels


Converting VisDrone2019-DET-val:   0%|          | 0/548 [00:00<?, ?it/s]

  → 548 label files written to datasets/VisDrone/VisDrone2019-DET-val/labels


Converting VisDrone2019-DET-test-dev:   0%|          | 0/1610 [00:00<?, ?it/s]

  → 1610 label files written to datasets/VisDrone/VisDrone2019-DET-test-dev/labels


## 3. Create data.yaml for Ultralytics

In [ ]:
yaml_content = f"""
path: {DATA_ROOT.resolve()}
train: VisDrone2019-DET-train/images
val: VisDrone2019-DET-val/images
test: VisDrone2019-DET-test-dev/images

names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor
"""

yaml_path = DATA_ROOT / "visdrone.yaml"
with open(yaml_path, "w") as f:
    f.write(yaml_content.strip())

print("Saved:", yaml_path)
print(yaml_content)

Saved: datasets/VisDrone/visdrone.yaml

path: /content/datasets/VisDrone
train: VisDrone2019-DET-train/images
val: VisDrone2019-DET-val/images
test: VisDrone2019-DET-test-dev/images

names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor



## 4. Baseline configuration (FIXED)

All parameters below are the **baseline**.  
We will change **only input resolution** later.

In [ ]:
# ========== BASELINE CONFIG ==========
BASELINE_CONFIG = {
    "model": "yolov8n.pt",          # pretrained COCO weights
    "model_source": "Ultralytics YOLOv8n (COCO pretrained)",
    "imgsz": 640,                  # input resolution
    "conf": 0.25,                  # confidence threshold
    "iou": 0.7,                    # NMS IoU threshold
    "eval_split": "val",           # VisDrone2019-DET-val
    "data_yaml": str(yaml_path),
    "hardware": HARDWARE,
    "seed": SEED,
}

print("=== BASELINE CONFIGURATION ===")
for k, v in BASELINE_CONFIG.items():
    print(f"  {k}: {v}")

=== BASELINE CONFIGURATION ===
  model: yolov8n.pt
  model_source: Ultralytics YOLOv8n (COCO pretrained)
  imgsz: 640
  conf: 0.25
  iou: 0.7
  eval_split: val
  data_yaml: datasets/VisDrone/visdrone.yaml
  hardware: GPU: Tesla T4
  seed: 42


## 5. Run Baseline (validation)

In [ ]:
model = YOLO(BASELINE_CONFIG["model"])

print("Running baseline validation...")
t0 = time.time()

baseline_results = model.val(
    data=BASELINE_CONFIG["data_yaml"],
    imgsz=BASELINE_CONFIG["imgsz"],
    conf=BASELINE_CONFIG["conf"],
    iou=BASELINE_CONFIG["iou"],
    split="val",
    plots=False,
    verbose=True,
)

baseline_time = time.time() - t0
print(f"\nBaseline validation finished in {baseline_time:.1f} s")

Running baseline validation...
Ultralytics 8.4.184 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1978.1±451.6 MB/s, size: 170.6 KB)
val: Scanning /content/datasets/VisDrone/VisDrone2019-DET-val/labels... 548 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 548/548 1.5Kit/s 0.4s
val: New cache created: /content/datasets/VisDrone/VisDrone2019-DET-val/labels.cache
WARNING ⚠️ Dataset images contain up to 317 objects (val=317), but max_det=300. This mismatch can cap recall and produce invalid validation metrics. Raising it may increase validation cost but cannot increase model or export capacity, which may cap recall. Setting max_det=317 to match the observed maximum.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 6.8it/s 5.1s
                   all        548      38759    

In [ ]:
# Extract key metrics
def extract_metrics(results):
    """Pull mAP, precision, recall from Ultralytics results."""
    metrics = {
        "mAP50-95": float(results.box.map),      # main metric
        "mAP50": float(results.box.map50),
        "precision": float(results.box.mp),      # mean precision
        "recall": float(results.box.mr),         # mean recall
    }
    return metrics

baseline_metrics = extract_metrics(baseline_results)

print("=== BASELINE METRICS ===")
for k, v in baseline_metrics.items():
    print(f"  {k}: {v:.4f}")

# Approximate memory (if GPU)
if torch.cuda.is_available():
    mem_allocated = torch.cuda.max_memory_allocated() / 1024**2
    print(f"  Peak GPU memory: {mem_allocated:.0f} MB")
    baseline_memory_mb = mem_allocated
else:
    baseline_memory_mb = None
    print("  (CPU mode — GPU memory not applicable)")

=== BASELINE METRICS ===
  mAP50-95: 0.0037
  mAP50: 0.0069
  precision: 0.0789
  recall: 0.0147
  Peak GPU memory: 274 MB


## 6. Small / Medium / Large object analysis

We use the same size definition as in the Dataset Audit (approximate side length = √area):
- **small**: side < 32 px
- **medium**: 32 ≤ side < 96 px
- **large**: side ≥ 96 px

Ultralytics also reports COCO-style small/medium/large AP. We show both.

In [ ]:
# COCO-style size metrics from Ultralytics (if available)
print("=== Size-based metrics (from Ultralytics / COCO style) ===")
try:
    # results.box.maps gives per-class; size breakdown may be in results
    print(f"  mAP50-95 (all): {baseline_results.box.map:.4f}")
    # Ultralytics stores area-based metrics in some versions
    if hasattr(baseline_results.box, "maps"):
        print("  Per-class mAP50-95:")
        for i, name in CLASS_NAMES.items():
            if i < len(baseline_results.box.maps):
                print(f"    {name}: {baseline_results.box.maps[i]:.4f}")
except Exception as e:
    print("  Could not extract detailed size metrics:", e)

# Manual count of GT objects by size on val set
def count_objects_by_size(split_dir: Path):
    labels_dir = split_dir / "labels"
    images_dir = split_dir / "images"
    counts = {"small": 0, "medium": 0, "large": 0, "total": 0}

    for label_path in labels_dir.glob("*.txt"):
        img_path = images_dir / (label_path.stem + ".jpg")
        if not img_path.exists():
            continue
        with Image.open(img_path) as im:
            W, H = im.size

        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                bw, bh = float(parts[3]), float(parts[4])
                side = np.sqrt(bw * W * bh * H)  # approx side in pixels
                counts["total"] += 1
                if side < 32:
                    counts["small"] += 1
                elif side < 96:
                    counts["medium"] += 1
                else:
                    counts["large"] += 1
    return counts

val_counts = count_objects_by_size(DATA_ROOT / "VisDrone2019-DET-val")
print("\n=== GT object counts on val (by size) ===")
for k, v in val_counts.items():
    print(f"  {k}: {v}")

=== Size-based metrics (from Ultralytics / COCO style) ===
  mAP50-95 (all): 0.0037
  Per-class mAP50-95:
    pedestrian: 0.0297
    people: 0.0000
    bicycle: 0.0000
    car: 0.0000
    van: 0.0000
    truck: 0.0059
    tricycle: 0.0000
    awning-tricycle: 0.0001
    bus: 0.0006
    motor: 0.0001

=== GT object counts on val (by size) ===
  small: 26586
  medium: 11105
  large: 1068
  total: 38759


## 7. Controlled change: increase input resolution

**What we change:** `imgsz` 640 → 1280  
**Why:** In DZ01 we saw that ~60% of objects have side < 32 px.  
Higher resolution should help the detector see small objects better.  
**What stays the same:** model weights, conf, iou, eval split, metric calculation.

In [ ]:
CHANGED_CONFIG = BASELINE_CONFIG.copy()
CHANGED_CONFIG["imgsz"] = 1280   # <-- the ONLY change

print("=== CONTROLLED CHANGE ===")
print(f"  What changed : imgsz {BASELINE_CONFIG['imgsz']} → {CHANGED_CONFIG['imgsz']}")
print(f"  Why          : improve detection of small objects (60% of boxes < 32 px)")
print(f"  Everything else stays identical.")

=== CONTROLLED CHANGE ===
  What changed : imgsz 640 → 1280
  Why          : improve detection of small objects (60% of boxes < 32 px)
  Everything else stays identical.


In [ ]:
# Clear GPU memory before second run
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

model2 = YOLO(CHANGED_CONFIG["model"])  # same weights

print("Running changed configuration (imgsz=1280)...")
t0 = time.time()

changed_results = model2.val(
    data=CHANGED_CONFIG["data_yaml"],
    imgsz=CHANGED_CONFIG["imgsz"],
    conf=CHANGED_CONFIG["conf"],
    iou=CHANGED_CONFIG["iou"],
    split="val",
    plots=False,
    verbose=True,
)

changed_time = time.time() - t0
print(f"\nChanged validation finished in {changed_time:.1f} s")

changed_metrics = extract_metrics(changed_results)

if torch.cuda.is_available():
    changed_memory_mb = torch.cuda.max_memory_allocated() / 1024**2
else:
    changed_memory_mb = None

print("\n=== CHANGED METRICS ===")
for k, v in changed_metrics.items():
    print(f"  {k}: {v:.4f}")
if changed_memory_mb:
    print(f"  Peak GPU memory: {changed_memory_mb:.0f} MB")

Running changed configuration (imgsz=1280)...
Ultralytics 8.4.184 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2993.9±926.2 MB/s, size: 111.3 KB)
val: Scanning /content/datasets/VisDrone/VisDrone2019-DET-val/labels.cache... 548 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 548/548 91.9Mit/s 0.0s
WARNING ⚠️ Dataset images contain up to 317 objects (val=317), but max_det=300. This mismatch can cap recall and produce invalid validation metrics. Raising it may increase validation cost but cannot increase model or export capacity, which may cap recall. Setting max_det=317 to match the observed maximum.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 3.4it/s 10.2s
                   all        548      38759     0.0832     0.0428     0.0203      0.011
                perso

## 8. Before / After comparison table

In [ ]:
comparison = pd.DataFrame({
    "Metric": ["mAP50-95", "mAP50", "Precision", "Recall",
               "Processing time (s)", "Peak GPU memory (MB)"],
    "Baseline (imgsz=640)": [
        f"{baseline_metrics['mAP50-95']:.4f}",
        f"{baseline_metrics['mAP50']:.4f}",
        f"{baseline_metrics['precision']:.4f}",
        f"{baseline_metrics['recall']:.4f}",
        f"{baseline_time:.1f}",
        f"{baseline_memory_mb:.0f}" if baseline_memory_mb else "N/A",
    ],
    "Changed (imgsz=1280)": [
        f"{changed_metrics['mAP50-95']:.4f}",
        f"{changed_metrics['mAP50']:.4f}",
        f"{changed_metrics['precision']:.4f}",
        f"{changed_metrics['recall']:.4f}",
        f"{changed_time:.1f}",
        f"{changed_memory_mb:.0f}" if changed_memory_mb else "N/A",
    ],
})

print("=== BEFORE / AFTER ===")
display(comparison)

# Delta
delta_map = changed_metrics["mAP50-95"] - baseline_metrics["mAP50-95"]
print(f"\nΔ mAP50-95: {delta_map:+.4f}")
print(f"Time ratio: {changed_time / baseline_time:.2f}x")
if baseline_memory_mb and changed_memory_mb:
    print(f"Memory ratio: {changed_memory_mb / baseline_memory_mb:.2f}x")

=== BEFORE / AFTER ===


,Metric,Baseline (imgsz=640),Changed (imgsz=1280)
0,mAP50-95,0.0037,0.0110
1,mAP50,0.0069,0.0203
2,Precision,0.0789,0.0832
3,Recall,0.0147,0.0428
4,Processing time (s),14.5,10.5
5,Peak GPU memory (MB),274,992



Δ mAP50-95: +0.0074
Time ratio: 0.72x
Memory ratio: 3.62x


## 9. Summary for the Report

Copy these numbers into the Detection Baseline Report.

In [ ]:
print("=" * 60)
print("SUMMARY FOR DETECTION BASELINE REPORT")
print("=" * 60)

print("\n1. RUN CONDITIONS (Baseline)")
for k, v in BASELINE_CONFIG.items():
    print(f"   {k}: {v}")

print("\n2. BASELINE METRICS")
for k, v in baseline_metrics.items():
    print(f"   {k}: {v:.4f}")

print("\n3. OBJECT COUNTS ON VAL (by size)")
for k, v in val_counts.items():
    print(f"   {k}: {v}")

print("\n4. CONTROLLED CHANGE")
print("   What : imgsz 640 → 1280")
print("   Why  : ~60% of objects are small (<32 px side)")

print("\n5. BEFORE / AFTER")
display(comparison)

print("\nDone. Use these numbers in the report.")

SUMMARY FOR DETECTION BASELINE REPORT

1. RUN CONDITIONS (Baseline)
   model: yolov8n.pt
   model_source: Ultralytics YOLOv8n (COCO pretrained)
   imgsz: 640
   conf: 0.25
   iou: 0.7
   eval_split: val
   data_yaml: datasets/VisDrone/visdrone.yaml
   hardware: GPU: Tesla T4
   seed: 42

2. BASELINE METRICS
   mAP50-95: 0.0037
   mAP50: 0.0069
   precision: 0.0789
   recall: 0.0147

3. OBJECT COUNTS ON VAL (by size)
   small: 26586
   medium: 11105
   large: 1068
   total: 38759

4. CONTROLLED CHANGE
   What : imgsz 640 → 1280
   Why  : ~60% of objects are small (<32 px side)

5. BEFORE / AFTER


,Metric,Baseline (imgsz=640),Changed (imgsz=1280)
0,mAP50-95,0.0037,0.0110
1,mAP50,0.0069,0.0203
2,Precision,0.0789,0.0832
3,Recall,0.0147,0.0428
4,Processing time (s),14.5,10.5
5,Peak GPU memory (MB),274,992



Done. Use these numbers in the report.
